Data Generator

In [10]:
import numpy as np
import tensorflow as tf
import os
import math
import json

# Load metadata
with open('../../../dataset/vocab.json', 'r') as f:
    vocab_data = json.load(f)

VOCAB_SIZE      = vocab_data['vocab_size']
MAX_LENGTH      = vocab_data['max_length']
WORD_TO_INDEX   = vocab_data['word_to_index']
FEATURE_DIR     = "../../../dataset/features"

In [11]:
class CaptionDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, batch_size, feature_dir, word_to_index, max_length):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.feature_dir = feature_dir
        self.word_to_index = word_to_index
        self.max_length = max_length

    def __len__(self):
        # Jumlah step/batch dalam satu epoch
        return math.ceil(len(self.df) / self.batch_size)

    def __getitem__(self, idx):
        # Slicing dataframe untuk batch ini
        batch_df = self.df.iloc[idx * self.batch_size : (idx + 1) * self.batch_size]
        
        features_input = []
        captions_input = []
        targets = []
        
        for _, row in batch_df.iterrows():
            feat_path = os.path.join(self.feature_dir, row['image'] + ".npy")
            feature = np.load(feat_path)
            
            # Tokenize Caption
            seq = [self.word_to_index.get(w, 0) for w in str(row['caption']).split()]
            
            in_seq = seq[:-1]   # Input sequence: <start> ... SN-1
            out_seq = seq[1:]   # Target sequence: S0 ... <end>
            
            features_input.append(feature)
            captions_input.append(in_seq)
            targets.append(out_seq)
            
        # Padding
        captions_input = tf.keras.preprocessing.sequence.pad_sequences(
            captions_input, maxlen=self.max_length, padding='post'
        )
        targets = tf.keras.preprocessing.sequence.pad_sequences(
            targets, maxlen=self.max_length, padding='post'
        )
        
        inputs = {
            "image_input": np.array(features_input),
            "caption_input": np.array(captions_input)
        }
        
        return inputs, np.array(targets)

Build RNN Model

In [12]:
def build_rnn_model(num_layers, hidden_size, embed_dim=256):
    # Input 1: CNN Feature
    image_input = tf.keras.Input(shape=(2048,), name="image_input")
    image_emb = tf.keras.layers.Dense(embed_dim, activation='relu')(image_input)
    image_emb = tf.keras.layers.Reshape((1, embed_dim))(image_emb)

    # Input 2: Caption Sequence
    caption_input = tf.keras.Input(shape=(MAX_LENGTH,), name="caption_input")
    caption_emb = tf.keras.layers.Embedding(VOCAB_SIZE, embed_dim, mask_zero=True)(caption_input)

    # Pre-inject
    merged = tf.keras.layers.Concatenate(axis=1)([image_emb, caption_emb])

    # Recurrent Layers
    x = merged
    for i in range(num_layers):
        x = tf.keras.layers.SimpleRNN(hidden_size, return_sequences=True)(x)

    # Output Layer -> timestep teks = index 1 dst.
    x = tf.keras.layers.Lambda(lambda t: t[:, 1:, :])(x)
    output = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(VOCAB_SIZE, activation='softmax'))(x)

    model = tf.keras.Model(inputs=[image_input, caption_input], outputs=output)
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

Training Loop

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Data Preparation
df_captions = pd.read_csv("../../../dataset/captions.txt")
df_captions['caption'] = "<start> " + df_captions['caption'].str.lower() + " <end>"
train_df, val_df = train_test_split(df_captions, test_size=0.1, random_state=42)

# Hyperparameters
layers_variations = [1, 2, 3]
hidden_variations = [256, 512]

In [16]:
# Instansiasi Generator
train_gen = CaptionDataGenerator(
    df=train_df, 
    batch_size=128, 
    feature_dir=FEATURE_DIR, 
    word_to_index=WORD_TO_INDEX, 
    max_length=MAX_LENGTH
)

val_gen = CaptionDataGenerator(
    df=val_df, 
    batch_size=128, 
    feature_dir=FEATURE_DIR, 
    word_to_index=WORD_TO_INDEX, 
    max_length=MAX_LENGTH
)

for n_layers in layers_variations:
    for h_size in hidden_variations:
        version_name = f"rnn_l{n_layers}_h{h_size}"
        save_path = f"../weights/"
        os.makedirs(save_path, exist_ok=True)
        
        print(f"\nTraining Variasi: {version_name}")
        
        model = build_rnn_model(num_layers=n_layers, hidden_size=h_size)
        
        # Training
        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=1
        )
        
        # Simpan bobot
        model.save_weights(os.path.join(save_path, f"{version_name}.weights.h5"))
        print(f"Bobot disimpan di {save_path}")


Training Variasi: rnn_l1_h256


d:\GitHub\CNN-RNN-LSTM-Scratch\env\Lib\site-packages\keras\src\layers\layer.py:1035: UserWarning: Layer 'lambda_6' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
d:\GitHub\CNN-RNN-LSTM-Scratch\env\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
d:\GitHub\CNN-RNN-LSTM-Scratch\env\Lib\site-packages\keras\src\layers\layer.py:1035: UserWarning: Layer 'lambda_6' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstrea

285/285 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8562 - loss: 2.1435

d:\GitHub\CNN-RNN-LSTM-Scratch\env\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
d:\GitHub\CNN-RNN-LSTM-Scratch\env\Lib\site-packages\keras\src\layers\layer.py:1035: UserWarning: Layer 'lambda_6' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


285/285 ━━━━━━━━━━━━━━━━━━━━ 756s 3s/step - accuracy: 0.8837 - loss: 1.1436 - val_accuracy: 0.8933 - val_loss: 0.7083
Bobot disimpan di ../weights/

Training Variasi: rnn_l1_h512


...

285/285 ━━━━━━━━━━━━━━━━━━━━ 1094s 4s/step - accuracy: 0.8814 - loss: 1.1761 - val_accuracy: 0.8892 - val_loss: 0.8365
Bobot disimpan di ../weights/

Training Variasi: rnn_l3_h512


d:\GitHub\CNN-RNN-LSTM-Scratch\env\Lib\site-packages\keras\src\layers\layer.py:1035: UserWarning: Layer 'lambda_11' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
d:\GitHub\CNN-RNN-LSTM-Scratch\env\Lib\site-packages\keras\src\layers\layer.py:1035: UserWarning: Layer 'lambda_11' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


285/285 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.8604 - loss: 1.5076

d:\GitHub\CNN-RNN-LSTM-Scratch\env\Lib\site-packages\keras\src\layers\layer.py:1035: UserWarning: Layer 'lambda_11' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


285/285 ━━━━━━━━━━━━━━━━━━━━ 1533s 5s/step - accuracy: 0.8834 - loss: 1.0197 - val_accuracy: 0.8892 - val_loss: 0.8215
Bobot disimpan di ../weights/
